# HeatUp 01 - prepare

Puts every structure on disk and writes down **what is to be computed** for it. No calculator, no GPU.

| step | output (per `<material>/<label>/`) | used by | what it does |
|---|---|---|---|
| `relax` | `relaxation/CONTCAR`, `energy.json` | everything | BFGS (+ FrechetCellFilter) geometry optimisation |
| `elastic` | `elastic/elastic.json` (relaxed-ion) | mechanical | 6x6 Voigt stiffness tensor from strained cells |
| `harmonic` | `phonons/harmonic/harmonic.json` | vibrational, thermodynamic | second-order force constants at the 0 K cell (phonopy finite displacements): bands, DOS, free energy; also the competing-phase free energies |
| `quasiharmonic` (optional) | `phonons/quasiharmonic/quasiharmonic.json` | vibrational, thermodynamic with `cfg.terms.vibrational="quasiharmonic"` | harmonic phonons at several volumes, F(V,T) fit |
| `perturbative` (optional) | `phonons/perturbative/perturbative.json` | diagnostic | perturbative anharmonic: third-order force constants (phono3py), kappa_RTA(T), linewidths |
| `md` | `md/<T>K/output.traj` (positions, momenta, energy, stress) | structural transition, vibrational (anharmonic); thermodynamic with `cfg.terms.vibrational="anharmonic"` (then also for the competing phases) | NPT or NVT trajectory in a supercell |

Everything below is a *calculation* setting. Together with the gates you ask for (`cfg.plan.gates`) and the steps they
need, it is written into a `plan.json` next to every POSCAR, so the folders describe themselves:

* the structures are **relaxed here** if you ask for `"relax"` (otherwise the given ones are used as they are), and from them the
  **cells of every calculation are written into the folders** - phonon supercell and displacements, strained cells, the MD supercell,
  quasi-harmonic volumes, perturbative (fc3) displacements - so a step can just as well be computed with **another code** (DFT, ...);
* **notebook 02** (optional) loads the interatomic potential and runs the rest of what the plans ask for;
* **notebook 03** analyses whatever is on disk - also results you computed yourself (DFT, another potential).

The calculator, the gate thresholds and the analysis options are **not** set here.

## Where the data lives
```
<candidates_root>/<formula>/<label>/POSCAR
<database_root>/<material>/<label>/
    POSCAR
    plan.json
    metadata.json
    relaxation/
    elastic/
    phonons/            one folder per vibrational level, all its files inside:
        harmonic/  quasiharmonic/  perturbative/  anharmonic/   (anharmonic has no calculation: it holds the
                                                                post-processing of md/ and says so in source.json)
    md/<T>K/
    stability/
    <step dir>/step_<step>.json
```
A structure is **copied** from candidates to the database the moment it is needed (as a target, or as a competing phase of a target).
Competing phases already in the database are reused - but only if they were calculated with the settings of this project;
otherwise they are left untouched, a warning is printed and they are not used.
`print_layout(root)` (end of this notebook) shows what exists. The full description is the docstring of
`heatup.core.layout`.

In [ ]:
import json
import os
import warnings
import pandas as pd
from IPython.display import display

import heatup
from heatup import cfg, export_inputs, prepare_project, preview_plan, run_plan, save_project
from heatup.analyze.report import print_layout, show_status

warnings.filterwarnings("ignore")
heatup.verbose()
print("HeatUp", heatup.__version__)

## Configuration - the only cell you normally edit
Sections of `cfg` used here: `phases`, `plan`, `relax`, `elastic`, `harmonic`, `quasiharmonic`, `perturbative`, `md`, `terms`, `vibrational` (and `generation`, `phases.mp_*` for competing phases).

In [ ]:
# -- The two folders ------------------------------------------------------------------------------------
cfg.phases.database_root = "database"  # everything computed: targets and competing phases
cfg.phases.candidates_root = "candidates"  # structures only (yours; MP downloads and generated ones are added here)
# Competing phases are searched in both folders; "mp-api" (needs cfg.phases.mp_api_key or $MP_API_KEY) and "generate" add structures
cfg.phases.sources = ["database", "candidates"]  # ["database", "candidates", "mp-api"]

# -- Calculator: used in THIS notebook only to relax (if "relax" is in cfg.plan.gates); saved for notebook 02, which uses the same
cfg.calc.backend = os.environ.get("HEATUP_BACKEND", "mace-mp")      # 'mace-mp' | 'chgnet' | 'm3gnet' | 'custom'
cfg.calc.model = os.environ.get("MACE_MODEL_PATH", "medium-mpa-0")   # mace_mp name or path to a .model
cfg.calc.model_overrides = {}        # per-material fine-tuned model, e.g. {"BaTiO3": "BTO_MACE.model"}
cfg.calc.dispersion = False          # D3 correction on top of the MLIP
cfg.calc.dtype = "float64"           # keep float64 for finite-difference phonons / elastic constants
DEVICE = None                        # None -> cuda if usable, else cpu

# -- WHAT to compute: ask for what you want, everything it needs is planned (and notebook 02 runs all of it) ----
#   "relax"    -> optimise the structures. LEAVE IT OUT if they are already relaxed (other code / model / DFT): they are
#                 then used as given and only energy, forces and stress are evaluated (needed for the hull)
#   mechanical -> elastic | structural_transition -> md | vibrational -> the steps of cfg.vibrational.sources
#   thermodynamic -> the steps of the selected vibrational model, for the target AND its competing phases
cfg.plan.gates = [
    "relax",
    "mechanical",
    "structural_transition",
    "vibrational",
    "thermodynamic"
    ]
# -- Evidence of the vibrational check (what it plans, reads and plots; any subset, in any order) --------------------
#   "harmonic"      phonon bands / DOS at the 0 K cell: imaginary branches are listed (which branch, which q-point) and drawn in red
#   "quasiharmonic" the same at the volumes the crystal visits up to the highest temperature
#   "anharmonic"    VDOS of the MD runs: weight at w -> 0 = soft modes (a VDOS has no negative frequencies)
# The most complete evidence that exists decides; the others are reported as information.
cfg.vibrational.sources = ["harmonic", "anharmonic"]
# -- Terms of the hull's free energy (heatup.core.contributions) ------------------------------------------------
# G(T) = H0 (the relaxed 0 K energy, always there) + the terms you switch on below.  ONE choice per term, the SAME for every
# phase.  It decides the calculations of EVERY phase (targets AND competitors) that notebook 02 runs; notebook 03 follows it
# unless it is overridden there.  cfg.terms.only(...) switches on what you list and everything else off.
cfg.terms.vibrational     = "quasiharmonic"  # lattice vibrations: "off" = static 0 K hull | "ZPE" zero-point energy only
                                     #   | "harmonic" (2nd-order force constants at the 0 K cell) | "quasiharmonic" (+ volume effects)
                                     #   | "anharmonic" (phonons from the MD; experimental)
cfg.terms.electronic      = "dos"    # electronic entropy (metals): "off" | "dos" (needs electronic/edos.json in each phase)
cfg.terms.magnetic        = "ihj"    # magnetic entropy: "off" | "ihj" (needs magnetic/moments.json in each phase)
cfg.terms.configurational = "ideal"  # disorder: "off" | "ideal" (-T S of ideal mixing) | "enumerated" | "monte_carlo"
cfg.terms.orientational   = "off"    # hindered rotors / cation orientations: "off" | "rotors" (needs disorder/rotors.json)
cfg.terms.defects         = "off"    # composition-conserving defect pairs: "off" | "pairs" (needs disorder/defects.json)
# e.g. a hull of configurational entropy only:  cfg.terms.only(configurational="ideal")
# The settings of a term live in the config sections listed in the table printed below (cfg.harmonic, cfg.quasiharmonic, cfg.md, ...).
# (the target's own MD, for the structural-transition and vibrational checks, is planned by the gates)
# Optional: list the steps yourself instead (relax | elastic | phonons | qha | fc3 | md; relax is always added),
# e.g. to add "qha" / "fc3", or "md" for competitors (VDOS free energies). None = derived from the gates.
cfg.plan.target_steps     = None
cfg.plan.competitor_steps = None  # derived: relax + the steps of the selected vibrational model; add "md" for MD folders in every competing phase too
cfg.plan.md_temperatures  = None  # None -> the 'temperature' of each run-list entry; or e.g. [300, 600]
cfg.plan.prune_window_eV  = None  # optional cost saver, e.g. 0.15: competitors that far above the 0 K hull get no harmonic/MD steps

# -- Relaxation (used when "relax" is in cfg.plan.gates) ------------------------------------------------
cfg.relax.fmax = 0.005              # eV/A
cfg.relax.max_steps = 1000
cfg.relax.relax_cell = True         # relax cell shape and volume too (False: ions only, the given cell is kept)
cfg.relax.constant_volume = False   # keep the volume, relax shape + ions
cfg.relax.fix_symmetry = True       # keep the space group during the relaxation
cfg.relax.symprec = 1e-3
cfg.relax.pressure_GPa = 0.0

# -- HARMONIC: second-order force constants (phonopy, finite displacements) ---------------------------------------------------
# Sizing: None = "as needed". An explicit supercell wins (then leave min_length_A None); otherwise the smallest
# supercell with perpendicular widths >= min_length_A is built (None -> built-in 12 A); max_atoms None = no limit.
cfg.harmonic.supercell = (3, 3, 3)         # e.g. (3, 3, 3)
cfg.harmonic.min_length_A = None      # e.g. 12.0
cfg.harmonic.max_atoms = None         # e.g. 800 as a guard against runaway cells
cfg.harmonic.delta_A = 0.01           # displacement amplitude
cfg.harmonic.mesh_length_A = 80.0     # q-mesh density for the DOS
cfg.harmonic.band_npoints = 61
cfg.harmonic.symprec = 1e-4
cfg.harmonic.imag_tol_meV = 0.5       # imaginary modes shallower than this count as numerical noise
cfg.harmonic.fc_order = 2             # 3 = also third-order constants (phono3py, very slow); then add "fc3" to the steps

# -- PERTURBATIVE ANHARMONIC: third-order force constants (only with "perturbative" in the steps) -----------------------------------------
cfg.perturbative.supercell = (3, 3, 3)            # e.g. (2, 2, 2); same rule as above (None -> built-in 8 A)
cfg.perturbative.min_length_A = None
cfg.perturbative.max_atoms = None            # e.g. 400
cfg.perturbative.delta_A = 0.03
cfg.perturbative.cutoff_pair_A = None        # ignore atom pairs farther than this (e.g. 4.0): far fewer displaced cells
cfg.perturbative.separate_fc2 = True # fc2 from the harmonic supercell (cfg.harmonic), not the small fc3 one
cfg.perturbative.kappa_mesh_length_A = 30.0
cfg.perturbative.temperatures = [200.0, 400.0, 600.0, 800.0, 1000.0]
cfg.perturbative.min_freq_THz = 0.5
cfg.perturbative.ratio_warn = 0.05

# -- QUASI-HARMONIC (only with "quasiharmonic" in the steps) -----------------------------------------------------------------
cfg.quasiharmonic.n_volumes = 7               # odd, >= 5 recommended
cfg.quasiharmonic.volume_range = 0.06         # +/- fraction of V0
cfg.quasiharmonic.poly_order = 4              # polynomial order of the F(V) fit at each T
cfg.quasiharmonic.relax_internal = True       # relax ions at every volume (required)
cfg.quasiharmonic.temperatures = [200.0, 400.0, 600.0, 800.0, 1000.0]  # G(T) grid; the hull grid of notebook 03 must lie inside it

# -- Elastic tensor -------------------------------------------------------------------------------------
cfg.elastic.strain = 0.004
cfg.elastic.relax_ions = True       # relaxed-ion constants (clamped ions over-stiffen ferroelectrics)
cfg.elastic.fmax = 0.002

# -- MD -------------------------------------------------------------------------------------------------
cfg.md.ensemble = "NPT"  # 'NPT' (cell fluctuates) | 'NVT' (fixed cell)
cfg.md.timestep_fs = 1.0
cfg.md.n_steps = 300  # 30 ps at 1 fs; the analysis wants >= gates.min_production_ps after equilibration
cfg.md.nblock = 5  # a frame every 5 steps -> VDOS Nyquist 413 meV (20 would alias O-containing oxides)
cfg.md.ttime_fs = 50.0  # thermostat time constant
cfg.md.ptime_fs = 100.0  # barostat time constant (NPT)
cfg.md.pressure_GPa = 0.0
cfg.md.supercell = (2, 2, 2)  # e.g. (4, 4, 4); explicit wins, min_cell_A then stays None
cfg.md.min_cell_A = None  # minimum supercell edge for MD; None -> built-in 15 A
cfg.md.max_atoms = None  # e.g. 1500
cfg.md.seed = 12345

# -- Run list -------------------------------------------------------------------------------------------
USE_QUERY_FILE = False
QUERY_FILE = "next_candidates.json"   # [{"material": ..., "symmetry": ..., "temperature": ...}, ...]
MANUAL_LIST = [                       # "temperature": operating T and (unless cfg.plan.md_temperatures) MD T; may be a list
    {"material": "BaTiO3", "symmetry": "mp-1076932", "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-5020",    "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-5777",    "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-644497",  "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-995191",  "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-5933",    "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-5986",    "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-504715",  "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-558125",  "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-19990",   "temperature": 300},
    {"material": "BaTiO3", "symmetry": "mp-2998",    "temperature": 300},
]

### The selected terms
What the choices above switch on, the calculations notebook 02 will run for every phase, the rules that apply (double counting) and the current values of the settings of each selected term.

In [ ]:
from heatup.core import contributions
contributions.show()   # selection | steps planned for every phase | rules | settings of each selected term

## Prepare
A POSCAR in `database/<material>/<label>/` is used as is; one found only under `input/candidates/` is copied into the
database. Competing phases of every chemical subsystem are looked up in the database and in the candidates; the ones from
the candidates are copied into the database, and a `plan.json` plus the folders of all planned calculations are written for
everything. Competing phases that are already
in the database with other settings are **refused** (see `prepared.refused`). Safe to repeat: when an option changed,
notebook 02 sees the affected steps as *stale*.

In [ ]:
run_list = json.load(open(QUERY_FILE)) if USE_QUERY_FILE else MANUAL_LIST
prepared = prepare_project(run_list)            # competitors=False: targets only
if prepared.missing:
    print("missing structures:", prepared.missing)
for tag, conflicts in prepared.refused.items():       # already in the database, calculated with other settings: not used
    print("REFUSED", tag, {k: f"{a} -> {b}" for k, (a, b) in conflicts.items()})

## What will be computed
Supercells are taken from the *unrelaxed* structure (they can change by a repetition after relaxation) - check them before spending GPU time.

In [ ]:
display(pd.DataFrame(preview_plan(prepared)).set_index("phase"))
show_status()  # everything is pending until notebook 02 (or your own code) has run

## Relax, then write the cells
* **`"relax"` asked for** (`cfg.plan.gates`): every structure is relaxed now with the calculator above, so that `relaxation/CONTCAR`
  exists in all folders (finished relaxations are not repeated).
* **not asked for**: nothing is calculated; `CONTCAR` is a copy of the given `POSCAR` (already relaxed with another code or model).

From the `CONTCAR`s, `export_inputs` writes the structures of every planned calculation into its folder, as VASP `POSCAR` files:
`phonons/harmonic/` (phonopy supercell + displacements, `phonopy_disp.yaml`), `elastic/strain/` (12 strained cells), `md/POSCAR` (supercell),
`phonons/quasiharmonic/` (volumes), `phonons/perturbative/` (phono3py displacements). Notebook 02 builds exactly the same cells, so DFT and the
potential always see the same structures. Running it again changes nothing unless a structure or an option changed.

In [ ]:
if "relax" in cfg.plan.gates:
    relaxed = run_plan(steps=["relax"], device=DEVICE)
    display(pd.DataFrame(relaxed).groupby("action").size().rename("relaxations").to_frame())

exported = export_inputs(prepared)
display(pd.DataFrame(exported).set_index("phase"))

## The folders
One folder per planned calculation, each with the `POSCAR` it starts from, a `step_<step>.json` (what the step needs and which files it writes; the settings are in the phase's `plan.json`, and only there) and, after the export above,
the cells to compute. To compute a step with **another code** (e.g. DFT), run it on those cells and bring the results back:

* harmonic: forces (eV/A) of every `phonons/harmonic/disp-*` cell -> `heatup.collect_harmonic(phase, forces, perfect_forces=...)` writes `harmonic.json`
  (or any code that writes the files of notebook 03's last cell);
* elastic: stresses (GPa) of the strained cells, ions relaxed at fixed cell -> `heatup.collect_elastic(phase, {"xx_plus": [...], ...})`;
* MD: an `output.traj` (ASE, with momenta) and `md.json` in `md/<T>K/`; energies/structures: `relaxation/energy.json` and `CONTCAR`.

Notebook 02 is then not needed for that step, and notebook 03 treats the results as `external`.

In [ ]:
save_project()          # heatup_project.json: folder roots + calculator, so notebooks 02 and 03 find this project
print_layout(cfg.phases.database_root, max_depth=3)
print(open(os.path.join(cfg.phases.database_root, prepared.targets[0].tag, "elastic", "step_elastic.json")).read())